# Этап 4. Матричная факторизация SVD

## Цель работы

Реализовать рекомендательную модель на основе матричной факторизации
для предсказания пользовательских оценок
и формирования персональных Top-N рекомендаций.

В рамках этапа необходимо:

1. Подготовить обучающие данные для модели SVD.
2. Обучить модель матричной факторизации.
3. Реализовать предсказание оценок пользователей.
4. Сформировать Top-N персональных рекомендаций.
5. Исследовать основные параметры и ограничения модели.

Для обучения используем ту же обучающую выборку, что и в предыдущих этапах.
Тестовая выборка будет использоваться только для оценки качества.

In [1]:
# === Импорты ===

from time import perf_counter

import numpy as np
import pandas as pd

from surprise import accuracy

from homework_tasks.recommendation_systems.homework_goodbooks.src.config import (
    RANDOM_STATE,
    TRAIN_RATINGS_PATH,
    TEST_RATINGS_PATH,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.data_loader import (
    load_books,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.svd_model import (
    train_svd_model,
    predict_svd_rating,
    get_svd_recommendations,
)

In [2]:
# === Загрузка данных для матричной факторизации ===

train_ratings = pd.read_csv(TRAIN_RATINGS_PATH)
books = load_books()

print(f"Размер обучающей выборки: {train_ratings.shape}")
print(f"Количество пользователей: {train_ratings['user_id'].nunique()}")
print(f"Количество книг: {train_ratings['book_id'].nunique()}")
print(f"Диапазон оценок: {train_ratings['rating'].min():.1f}–"
      f"{train_ratings['rating'].max():.1f}")

display(train_ratings.head())

Размер обучающей выборки: (788503, 3)
Количество пользователей: 53424
Количество книг: 10000
Диапазон оценок: 1.0–5.0


,user_id,book_id,rating
0,9706,7475,1.0
1,41735,6392,3.0
2,23774,7890,3.0
3,52379,7322,4.0
4,10408,7965,4.0


In [3]:
# === Проверка данных перед обучением SVD ===

required_columns = {"user_id", "book_id", "rating"}

assert required_columns.issubset(train_ratings.columns)
assert not train_ratings[list(required_columns)].isna().any().any()
assert train_ratings["rating"].between(1, 5).all()
assert not train_ratings.duplicated(["user_id", "book_id"]).any()

print("Данные готовы к обучению SVD.")

Данные готовы к обучению SVD.


### Выводы по подготовке данных

Для обучения SVD используется та же обучающая выборка, что и в предыдущих этапах.

Данные содержат идентификаторы пользователей, идентификаторы книг и числовые оценки.

Проверены диапазон оценок, отсутствие пропущенных значений и повторяющихся пар пользователь–книга.

Тестовые взаимодействия не используются при обучении модели.

## 4.2. Обучение модели SVD

Обучим модель матричной факторизации с использованием библиотеки `scikit-surprise`.

Для начального эксперимента выберем 50 скрытых факторов и 20 эпох обучения.

Модель будет учитывать глобальную среднюю оценку, смещения пользователей и книг,
а также взаимодействие их скрытых факторов.

Измерим время обучения и проверим размерность полученных представлений.

In [4]:
# === Обучение модели SVD ===

n_factors = 50
n_epochs = 20

start_time = perf_counter()

svd_model = train_svd_model(
    train_ratings=train_ratings,
    n_factors=n_factors,
    n_epochs=n_epochs,
    random_state=RANDOM_STATE,
)

training_time_seconds = perf_counter() - start_time

print("Обучение SVD завершено.")
print(f"Количество скрытых факторов: {n_factors}")
print(f"Количество эпох: {n_epochs}")
print(f"Время обучения: {training_time_seconds:.2f} с")

Обучение SVD завершено.
Количество скрытых факторов: 50
Количество эпох: 20
Время обучения: 6.11 с


In [5]:
# === Проверка параметров обученной модели ===

print(f"Размерность факторов пользователей: {svd_model.pu.shape}")
print(f"Размерность факторов книг: {svd_model.qi.shape}")

print(f"Количество смещений пользователей: {len(svd_model.bu)}")
print(f"Количество смещений книг: {len(svd_model.bi)}")

print(f"Глобальная средняя оценка: {svd_model.trainset.global_mean:.4f}")

assert svd_model.pu.shape == (
    train_ratings["user_id"].nunique(),
    n_factors,
)
assert svd_model.qi.shape == (
    train_ratings["book_id"].nunique(),
    n_factors,
)

print("Размерности модели соответствуют обучающим данным.")

Размерность факторов пользователей: (53424, 50)
Размерность факторов книг: (10000, 50)
Количество смещений пользователей: 53424
Количество смещений книг: 10000
Глобальная средняя оценка: 3.8558
Размерности модели соответствуют обучающим данным.


### Выводы по обучению SVD

По результатам обучения модели матричной факторизации установлено:

1. Для обучения использована выборка из 788 503 пользовательских оценок.

2. Модель обучена с использованием 50 скрытых факторов и 20 эпох.

3. Время обучения составило 6,11 секунды.

4. Сформированы матрицы скрытых факторов пользователей размерностью 53 424 x 50
и книг размерностью 10 000 x 50.

5. Модель также содержит смещения для 53 424 пользователей и 10 000 книг.

6. Глобальная средняя оценка обучающей выборки составляет 3,8558.

7. Все проверки размерностей обученной модели завершились успешно.

**Итог:**

Обучена модель SVD, представляющая пользователей и книги в пространстве скрытых факторов.

В отличие от Item-Based Collaborative Filtering, модель не требует непосредственного расчета сходства книг при каждом предсказании оценки.

## 4.3. Предсказание пользовательских оценок

После обучения модель SVD позволяет прогнозировать оценку пользователя для книги на основе глобальной средней оценки,
смещений пользователя и книги, а также взаимодействия скрытых факторов.

Для получения прогноза используется метод `predict()` обученной модели.

Проверим результат на том же тестовом взаимодействии, которое ранее использовалось для Item-Based Collaborative Filtering:

- пользователь `12874`
- книга `Ulysses`
- фактическая оценка 2,0

Фактическая тестовая оценка будет использоваться только для сравнения с прогнозом.
При обучении модели она не использовалась.

Для неизвестных пользователей или книг наша функция возвращает `NaN`,
поскольку полноценное персонализированное предсказание в этом случае невозможно.

In [6]:
# === Предсказание оценки SVD на тестовом взаимодействии ===

test_ratings = pd.read_csv(TEST_RATINGS_PATH)

sample_user_id = 12874

sample_test_rating = test_ratings.loc[
    test_ratings["user_id"] == sample_user_id
    ].iloc[0]

sample_book_id = int(sample_test_rating["book_id"])
actual_rating = float(sample_test_rating["rating"])

predicted_rating = predict_svd_rating(
    model=svd_model,
    user_id=sample_user_id,
    book_id=sample_book_id,
)

book_title = books.loc[
    books["id"] == sample_book_id,
    "title",
].iloc[0]

print(f"ID пользователя: {sample_user_id}")
print(f"Книга: {book_title}")
print(f"Фактическая оценка: {actual_rating:.2f}")
print(f"Предсказанная оценка SVD: {predicted_rating:.2f}")
print(f"Абсолютная ошибка: {abs(actual_rating - predicted_rating):.2f}")

assert np.isfinite(predicted_rating)

ID пользователя: 12874
Книга: Ulysses
Фактическая оценка: 2.00
Предсказанная оценка SVD: 3.23
Абсолютная ошибка: 1.23


In [7]:
# === Проверка предсказаний для неизвестных идентификаторов ===

unknown_user_id = int(train_ratings["user_id"].max()) + 1
unknown_book_id = int(train_ratings["book_id"].max()) + 1

unknown_user_prediction = predict_svd_rating(
    model=svd_model,
    user_id=unknown_user_id,
    book_id=sample_book_id,
)

unknown_book_prediction = predict_svd_rating(
    model=svd_model,
    user_id=sample_user_id,
    book_id=unknown_book_id,
)

print(f"Неизвестный пользователь: {unknown_user_prediction}")
print(f"Неизвестная книга: {unknown_book_prediction}")

assert np.isnan(unknown_user_prediction)
assert np.isnan(unknown_book_prediction)

Неизвестный пользователь: nan
Неизвестная книга: nan


### Оценка RMSE на тестовой выборке

Для оценки точности предсказания пользовательских рейтингов
рассчитаем RMSE (Root Mean Squared Error).

Метрика характеризует среднее отклонение предсказанных оценок
от фактических с дополнительным штрафом за большие ошибки.

Расчет выполняется на всей отложенной тестовой выборке,
которая не использовалась при обучении SVD.

В отличие от Precision@K, Recall@K и nDCG@K, RMSE оценивает точность числовых прогнозов,
а не качество рекомендательного списка.

In [8]:
# === Оценка RMSE модели SVD на тестовой выборке ===

svd_testset = list(
    test_ratings[["user_id", "book_id", "rating"]].itertuples(
        index=False,
        name=None,
    )
)

svd_test_predictions = svd_model.test(svd_testset)

svd_rmse = accuracy.rmse(
    svd_test_predictions,
    verbose=False,
)

print(f"Количество тестовых оценок: {len(svd_test_predictions)}")
print(f"RMSE SVD: {svd_rmse:.4f}")

assert len(svd_test_predictions) == len(test_ratings)
assert np.isfinite(svd_rmse)

Количество тестовых оценок: 190975
RMSE SVD: 0.8416


### Выводы по предсказанию пользовательских оценок

По результатам проверки SVD установлено:

1. Реализована функция `predict_svd_rating()`,
которая возвращает прогнозируемую оценку пользователя для книги.

2. Для пользователя `12874` и книги `Ulysses` получен прогноз 3,23 при фактической оценке 2,00.
Абсолютная ошибка составила 1,23.

3. Для того же взаимодействия Item-Based Collaborative Filtering
получил прогноз 3,31 с абсолютной ошибкой 1,31.

4. В рассматриваемом примере SVD показал меньшую абсолютную ошибку,
однако одного взаимодействия недостаточно для сравнения качества алгоритмов.

5. Для неизвестных пользователей и книг функция возвращает `NaN`,
не используя резервный прогноз библиотеки Surprise.

6. Модель использует скрытые факторы и смещения,
полученные при обучении исключительно на train.

**Итог:**

SVD позволяет предсказывать пользовательские оценки на основе обученных скрытых представлений пользователей и книг.

Для объективного сравнения с другими рекомендательными алгоритмами
необходимо оценить качество модели на множестве тестовых взаимодействий.

## 4.4. Формирование Top-N рекомендаций

Реализуем получение персональных рекомендаций с использованием обученной модели SVD.

Алгоритм выполняет следующие действия:

1. Определяет внутренний индекс пользователя в модели Surprise.
2. Рассчитывает прогнозируемые оценки для всех известных книг на основе скрытых факторов и смещений.
3. Ограничивает прогнозы допустимым диапазоном от 1 до 5.
4. Исключает книги, уже оцененные пользователем в обучающей выборке.
5. Сортирует оставшиеся книги по убыванию прогнозируемой оценки.
6. Возвращает Top-N с названиями и авторами.

В отличие от Item-Based CF, модель SVD не требует повторного расчета косинусной близости книг при формировании рекомендаций.

In [9]:
# === Получение Top-10 рекомендаций с помощью SVD ===

sample_user_id = 12874

svd_recommendations = get_svd_recommendations(
    model=svd_model,
    user_id=sample_user_id,
    books=books,
    top_n=10,
)

print(f"ID пользователя: {sample_user_id}")
print(f"Количество рекомендаций: {len(svd_recommendations)}")

display(svd_recommendations)

ID пользователя: 12874
Количество рекомендаций: 10


,book_id,title,authors,predicted_rating
0,8109,"The Absolute Sandman, Volume One","Neil Gaiman, Mike Dringenberg, Chris Bachalo, ...",4.634474
1,1788,The Calvin and Hobbes Tenth Anniversary Book,Bill Watterson,4.589118
2,5580,The Calvin and Hobbes Lazy Sunday Book,Bill Watterson,4.580203
3,1308,A Court of Mist and Fury (A Court of Thorns an...,Sarah J. Maas,4.509385
4,6361,There's Treasure Everywhere: A Calvin and Hobb...,Bill Watterson,4.494718
5,9842,Humans of New York: Stories,Brandon Stanton,4.488801
6,6920,The Indispensable Calvin and Hobbes,Bill Watterson,4.468466
7,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,4.462568
8,1010,The Essential Calvin and Hobbes: A Calvin and ...,Bill Watterson,4.436872
9,8976,Rita Hayworth and Shawshank Redemption: A Stor...,Stephen King,4.423806


In [10]:
# === Проверка корректности Top-N рекомендаций ===

rated_book_ids = set(
    train_ratings.loc[
        train_ratings["user_id"] == sample_user_id,
        "book_id",
    ]
)

recommended_book_ids = set(svd_recommendations["book_id"])

print(
    "Уже оцененных книг в рекомендациях:",
    len(rated_book_ids & recommended_book_ids),
)

print(
    "Количество отсутствующих предсказаний:",
    svd_recommendations["predicted_rating"].isna().sum(),
)

print(
    "Минимальная предсказанная оценка:",
    svd_recommendations["predicted_rating"].min(),
)

print(
    "Максимальная предсказанная оценка:",
    svd_recommendations["predicted_rating"].max(),
)

assert len(svd_recommendations) == 10
assert not (rated_book_ids & recommended_book_ids)
assert svd_recommendations["book_id"].is_unique
assert svd_recommendations["predicted_rating"].between(1, 5).all()
assert svd_recommendations["predicted_rating"].is_monotonic_decreasing

Уже оцененных книг в рекомендациях: 0
Количество отсутствующих предсказаний: 0
Минимальная предсказанная оценка: 4.42380646688596
Максимальная предсказанная оценка: 4.634473655133371


In [11]:
# === Сравнение пакетных прогнозов с model.predict() ===

prediction_checks = []

for book_id in svd_recommendations["book_id"].head(5):
    batch_prediction = svd_recommendations.loc[
        svd_recommendations["book_id"] == book_id,
        "predicted_rating",
    ].iloc[0]

    single_prediction = predict_svd_rating(
        model=svd_model,
        user_id=sample_user_id,
        book_id=int(book_id),
    )

    prediction_checks.append(
        {
            "book_id": int(book_id),
            "batch_prediction": batch_prediction,
            "single_prediction": single_prediction,
            "difference": abs(batch_prediction - single_prediction),
        }
    )

prediction_checks_df = pd.DataFrame(prediction_checks)

display(prediction_checks_df)

assert np.allclose(
    prediction_checks_df["batch_prediction"],
    prediction_checks_df["single_prediction"],
    atol=1e-6,
)

,book_id,batch_prediction,single_prediction,difference
0,8109,4.634474,4.634474,0.0
1,1788,4.589118,4.589118,0.0
2,5580,4.580203,4.580203,0.0
3,1308,4.509385,4.509385,0.0
4,6361,4.494718,4.494718,0.0


### Выводы по формированию Top-N рекомендаций

По результатам формирования рекомендаций с помощью SVD установлено:

1. Реализована функция `get_svd_recommendations()`,
которая формирует персональные Top-N рекомендации на основе обученных скрытых факторов и смещений.

2. Для пользователя `12874` сформировано 10 рекомендаций с предсказанными оценками от 4,4238 до 4,6345.

3. Ранее оцененные пользователем книги исключены из рекомендаций.
Все полученные оценки находятся в допустимом диапазоне от 1 до 5.

4. Пакетные предсказания для пяти рекомендованных книг полностью совпали с результатами `model.predict()`,
что подтверждает корректность реализации вычислений.

5. В полученном Top-10 присутствует несколько сборников `Calvin and Hobbes`.
Это указывает на ограниченное разнообразие рекомендаций в рассмотренном примере.

6. В отличие от Item-Based Collaborative Filtering,
SVD использует предварительно обученные скрытые факторы и не требует повторного расчета сходства между книгами при каждом запросе.

**Итог:**

Реализовано формирование персональных Top-N рекомендаций с помощью SVD.

Модель позволяет эффективно рассчитывать прогнозы для всего каталога,
однако полученная выдача может концентрироваться вокруг отдельных произведений или серий.

Для дальнейшей интерпретации рассмотрим влияние параметров модели и обучающей статистики книг.

## 4.5. Анализ параметров и ограничений SVD

Качество матричной факторизации зависит от количества скрытых факторов,
числа эпох обучения, скорости обучения и регуляризации.

В текущем эксперименте модель использует:

- `n_factors=50` -> количество скрытых факторов
- `n_epochs=20` -> количество эпох обучения
- параметры регуляризации и скорости обучения по умолчанию
- `random_state=42` -> воспроизводимость результатов

При анализе Top-N рекомендаций обнаружено,
что несколько позиций занимают сборники `Calvin and Hobbes`.

Для интерпретации результата рассмотрим средние оценки рекомендованных книг,
количество обучающих оценок и смещения книг, полученные моделью.

Параметры модели пока не изменяем, чтобы не использовать итоговую тестовую выборку для их подбора.

In [12]:
# === Анализ обучающей статистики рекомендованных книг ===

book_rating_stats = (
    train_ratings.groupby("book_id")
    .agg(
        ratings_count=("rating", "count"),
        average_rating=("rating", "mean"),
    )
    .reset_index()
)

recommendation_analysis = svd_recommendations.merge(
    book_rating_stats,
    on="book_id",
    how="left",
    validate="one_to_one",
)

display(
    recommendation_analysis[
        [
            "book_id",
            "title",
            "predicted_rating",
            "ratings_count",
            "average_rating",
        ]
    ].round(4)
)

,book_id,title,predicted_rating,ratings_count,average_rating
0,8109,"The Absolute Sandman, Volume One",4.6345,78,4.7436
1,1788,The Calvin and Hobbes Tenth Anniversary Book,4.5891,81,4.7407
2,5580,The Calvin and Hobbes Lazy Sunday Book,4.5802,85,4.8118
3,1308,A Court of Mist and Fury (A Court of Thorns an...,4.5094,82,4.7439
4,6361,There's Treasure Everywhere: A Calvin and Hobb...,4.4947,76,4.7500
5,9842,Humans of New York: Stories,4.4888,71,4.6620
6,6920,The Indispensable Calvin and Hobbes,4.4685,73,4.8356
7,5207,The Days Are Just Packed: A Calvin and Hobbes ...,4.4626,80,4.7625
8,1010,The Essential Calvin and Hobbes: A Calvin and ...,4.4369,73,4.6301
9,8976,Rita Hayworth and Shawshank Redemption: A Stor...,4.4238,80,4.5750


In [13]:
# === Разложение прогнозируемых оценок SVD на компоненты ===

user_index = svd_model.trainset.to_inner_uid(sample_user_id)

book_indices = [
    svd_model.trainset.to_inner_iid(int(book_id))
    for book_id in svd_recommendations["book_id"]
]

recommendation_analysis["book_bias"] = svd_model.bi[book_indices]
recommendation_analysis["user_bias"] = svd_model.bu[user_index]

recommendation_analysis["latent_interaction"] = (
    svd_model.qi[book_indices] @ svd_model.pu[user_index]
)

recommendation_analysis["raw_prediction"] = (
    svd_model.trainset.global_mean
    + recommendation_analysis["user_bias"]
    + recommendation_analysis["book_bias"]
    + recommendation_analysis["latent_interaction"]
)

display(
    recommendation_analysis[
        [
            "title",
            "predicted_rating",
            "book_bias",
            "latent_interaction",
            "ratings_count",
        ]
    ].round(4)
)

assert np.allclose(
    recommendation_analysis["predicted_rating"],
    recommendation_analysis["raw_prediction"].clip(1, 5),
    atol=1e-6,
)

,title,predicted_rating,book_bias,latent_interaction,ratings_count
0,"The Absolute Sandman, Volume One",4.6345,0.8430,0.3253,78
1,The Calvin and Hobbes Tenth Anniversary Book,4.5891,0.9645,0.1585,81
2,The Calvin and Hobbes Lazy Sunday Book,4.5802,0.9128,0.2013,85
3,A Court of Mist and Fury (A Court of Thorns an...,4.5094,0.8358,0.2074,82
4,There's Treasure Everywhere: A Calvin and Hobb...,4.4947,0.8592,0.1694,76
5,Humans of New York: Stories,4.4888,0.8658,0.1569,71
6,The Indispensable Calvin and Hobbes,4.4685,0.9981,0.0042,73
7,The Days Are Just Packed: A Calvin and Hobbes ...,4.4626,0.9902,0.0062,80
8,The Essential Calvin and Hobbes: A Calvin and ...,4.4369,0.8270,0.1437,73
9,Rita Hayworth and Shawshank Redemption: A Stor...,4.4238,0.7258,0.2319,80


### Выводы по анализу SVD

По результатам анализа статистики рекомендованных книг и параметров модели установлено:

1. Все десять рекомендованных книг имеют высокие средние оценки в обучающей выборке -> от 4,5750 до 4,8356.

2. Количество обучающих оценок рекомендованных книг составляет от 71 до 85.

3. Все рекомендованные книги имеют положительные смещения `book_bias` в диапазоне приблизительно от 0,73 до 1,00.
Это увеличивает прогноз относительно глобальной средней оценки.

4. Вклад взаимодействия скрытых факторов `latent_interaction` различается между книгами.
Для `The Absolute Sandman, Volume One` он составляет 0,3253,
а для `The Indispensable Calvin and Hobbes` -> 0,0042.

5. Высокие позиции некоторых сборников `Calvin and Hobbes` в значительной степени связаны с положительными смещениями книг,
а не только с персональным взаимодействием скрытых факторов.

6. Проверка разложения прогнозов на глобальную среднюю,
смещения и взаимодействие скрытых факторов завершилась успешно.

7. Используемые параметры `n_factors=50` и `n_epochs=20` являются стартовыми.
Их оптимальность пока не подтверждена экспериментально.

**Итог:**

SVD сочетает общие закономерности оценивания книг с индивидуальными предпочтениями пользователей.

В рассмотренном примере на рекомендации заметно влияют положительные смещения высоко оцененных книг.
При этом скрытые факторы также вносят персональный вклад в итоговые прогнозы.

Для объективной оценки качества и подбора параметров необходимо использовать отдельную валидационную выборку,
не затрагивая финальный test.

## 4.6. Итоговые выводы по этапу 4

В ходе четвертого этапа реализована рекомендательная модель матричной факторизации SVD с использованием библиотеки `scikit-surprise`.

**1. Подготовка данных**

- Использована обучающая выборка из 788 503 пользовательских оценок.
- Проверены диапазон оценок, отсутствие пропусков и повторяющихся взаимодействий.
- Итоговая тестовая выборка не использовалась для обучения.

**2. Обучение модели**

- Модель обучена с 50 скрытыми факторами и 20 эпохами.
- Получены матрицы факторов пользователей размерностью 53 424 x 50 и книг размерностью 10 000 x 50.
- Глобальная средняя оценка составляет 3,8558.
- Время обучения составило около 6 секунд.

**3. Предсказание оценок**

- Реализована функция `predict_svd_rating()`.
- Для пользователя `12874` и книги `Ulysses` получен прогноз 3,23 при фактической оценке 2,00.
- Абсолютная ошибка составила 1,23.
- Для неизвестных пользователей и книг функция возвращает `NaN`.
- RMSE = 0,8416.

**4. Формирование Top-N рекомендаций**

- Реализована функция `get_svd_recommendations()`.
- Для выбранного пользователя получено 10 рекомендаций с прогнозами от 4,4238 до 4,6345.
- Ранее оцененные книги исключены из выдачи.
- Пакетные расчеты согласуются с методом `model.predict()`.

**5. Интерпретация модели**

- В рекомендациях заметно присутствие высоко оцененных книг.
- Все книги из рассмотренного Top-10 имеют положительные смещения.
- Взаимодействие скрытых факторов также влияет на прогнозы.
- Рекомендации могут концентрироваться вокруг отдельных серий или произведений.

**6. Ограничения**

- Качество модели зависит от количества и репрезентативности обучающих оценок.
- Для новых пользователей и книг отсутствуют обученные скрытые представления.
- Скрытые факторы не имеют непосредственной содержательной интерпретации.
- Гиперпараметры модели пока не оптимизированы.
- Высокая предсказанная оценка не гарантирует релевантности рекомендации.

**Общий вывод:**

SVD позволяет формировать персональные книжные рекомендации на основе скрытых факторов пользователей и книг.

В отличие от Item-Based Collaborative Filtering, модель использует предварительно обученные представления
и не требует расчета сходства книг при каждом запросе.